In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/content/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())

Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())

Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


In [3]:
import pandas as pd
import plotly.express as px

# Load dataset
df = pd.read_csv('netflix_catalogue.csv')



df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'


ratings_filter = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']

filtered_df = df[df['rating'].isin(ratings_filter)]


heatmap_data = (
    filtered_df
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='title_count')
)

# Pivot table for heatmap
heatmap_pivot = heatmap_data.pivot(
    index='rating',
    columns='decade',
    values='title_count'
).fillna(0)

# Sort decades chronologically
heatmap_pivot = heatmap_pivot[
    sorted(
        heatmap_pivot.columns,
        key=lambda x: int(x[:-1])
    )
]


fig = px.imshow(
    heatmap_pivot,
    text_auto=True,                 # Show values in cells
    color_continuous_scale='Blues', # Sequential colour scale
    aspect='auto',
    labels=dict(
        x='Release Decade',
        y='Content Rating',
        color='Number of Titles'
    ),
    title=(
        "TV-MA and TV-14 dominate Netflix releases in "
        "the 2010s and 2020s"
    )
)


fig.add_annotation(
    x='2010s',
    y='TV-MA',
    text='Highest concentration of titles',
    showarrow=True,
    arrowhead=2
)


fig.update_layout(
    title_x=0.5,
    xaxis_title='Release Decade',
    yaxis_title='Content Rating',
    font=dict(size=12),
    height=600,
    width=900
)


fig.show()

In [4]:

import plotly.graph_objects as go

movies = df[df['type'] == 'Movie']

adds = (movies
        .groupby('added_year')
        .size()
        .reset_index(name='count'))

adds = adds[(adds['added_year'] >= 2015) & (adds['added_year'] <= 2022)].copy()

x_vals = adds['added_year'].astype(str).tolist() + ['Total 2015-2022']
y_vals = adds['count'].tolist() + [int(adds['count'].sum())]
measures = ['relative']*len(adds) + ['total']



trace = go.Waterfall(
    x=list(range(len(x_vals))),      # 0,1,2,...8 — NOT year strings
    y=y_vals,
    measure=measures,
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),
    totals=dict(marker_color='#2E75B6'),
    texttemplate='%{y:,}',
    textposition='outside'
)

fig = go.Figure(data=[trace])
fig.update_layout(
    title='Netflix library grew more than 10x in 9 years',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    yaxis=dict(gridcolor='#EEEEEE', title='Titles Added'),
    xaxis=dict(
        title='Year', showgrid=False,
        tickmode='array',
        tickvals=list(range(len(x_vals))),
        ticktext=x_vals              # labels shown on axis
    ),
    margin=dict(l=60, r=40, t=55, b=40),
    showlegend=False,
    height=700,
    annotations=[dict(
        x=1, y=93,                   # x=1 = index of '2016'
        text='<b>Peak year</b>',
        showarrow=True, arrowhead=2,
        arrowcolor='#e63946',
        font=dict(color='#e63946', size=11, family='Arial'),
        ax=0, ay=-40
    )]
)
fig.show()
